# 백테스트 Colab 실행기

이 노트북은 기간, 비용, 레버리지, 기본 전략 옵션을 바꿔가며 백테스트를 실행할 수 있도록 만든 실행용 노트북입니다.

## 사용 방법

1. 아래 입력칸에서 원하는 조건을 설정합니다.
2. 셀을 위에서부터 순서대로 실행합니다.
3. 실행이 끝나면 결과표를 확인합니다.
4. 필요하면 마지막 셀에서 결과 CSV 파일을 zip으로 내려받을 수 있습니다.

주의:

- 변수명은 영어 그대로 두는 것을 권장합니다.
- 기본값 그대로 먼저 한 번 실행해본 뒤, 그 다음에 기간이나 비용 조건을 바꿔보시는 것을 추천합니다.


In [ ]:
REPO_URL = "https://github.com/snowballTQ/clean-backtest.git"
BRANCH = "main"
PACKAGE_SUBDIR = "."


## 입력값 설명

아래 입력칸에서 백테스트 조건을 직접 바꿀 수 있습니다.

- `START_DATE`: 백테스트 시작일입니다.
- `END_DATE`: 백테스트 종료일입니다.
- `BASE_SERIES`: 기준 시계열입니다. `ndx`, `spx`, `composite_splice`를 쉼표로 여러 개 넣을 수 있습니다.
- `LEVERAGES`: 테스트할 레버리지 배수입니다. 예: `2`, `3`, `2,3`
- `INCLUDE_BASE_SERIES`: 1배 기본 시계열 전략도 같이 볼지 여부입니다.
- `INCLUDE_ZERO_COST`: 비용을 뺀 참고용 결과도 같이 볼지 여부입니다.
- `INCLUDE_COST_ADJUSTED`: 비용 반영 결과를 포함할지 여부입니다.
- `RUN_BUYHOLD`: 장기보유 비교를 실행할지 여부입니다.
- `RUN_PRICE_SMA`: 가격과 이동평균의 관계를 이용한 타이밍 전략을 실행할지 여부입니다.
- `RUN_DUAL_SMA`: 빠른 이동평균과 느린 이동평균의 교차 전략을 실행할지 여부입니다.
- `PRICE_SMA_WINDOW`: 가격-vs-SMA 전략의 이동평균 길이입니다.
- `PRICE_ENTRY_CONFIRM_DAYS`: 가격-vs-SMA 전략의 진입 확인 일수입니다.
- `PRICE_EXIT_CONFIRM_DAYS`: 가격-vs-SMA 전략의 청산 확인 일수입니다.
- `FAST_SMA_WINDOW`: 듀얼 SMA 전략의 빠른 이동평균 길이입니다.
- `SLOW_SMA_WINDOW`: 듀얼 SMA 전략의 느린 이동평균 길이입니다.
- `CROSS_ENTRY_CONFIRM_DAYS`: 듀얼 SMA 전략의 진입 확인 일수입니다.
- `CROSS_EXIT_CONFIRM_DAYS`: 듀얼 SMA 전략의 청산 확인 일수입니다.
- `COMMISSION_RATE`: 편도 수수료율입니다. 예: `0.001` = 0.10%
- `TAX_RATE`: 세율입니다. 예: `0.22` = 22%
- `EXPENSE_RATIO`: 연 운용보수 가정값입니다.
- `BORROW_SPREAD`: 차입 스프레드 가정값입니다.
- `OUTPUT_NAME`: 결과 파일이 저장될 폴더 이름입니다.

처음에는 기본값 그대로 한 번 실행해본 뒤, 그 다음에 시작일/종료일, 기준 시계열, 레버리지, 이동평균 조건, 수수료 같은 값들을 바꿔보시는 것을 추천합니다.


In [ ]:
START_DATE = "1985-10-01" # @param {type:"date"}
END_DATE = "2026-03-09" # @param {type:"date"}
BASE_SERIES = "ndx,spx" # @param {type:"string"}
LEVERAGES = "2,3" # @param {type:"string"}
INCLUDE_BASE_SERIES = True # @param {type:"boolean"}
INCLUDE_ZERO_COST = False # @param {type:"boolean"}
INCLUDE_COST_ADJUSTED = True # @param {type:"boolean"}
RUN_BUYHOLD = True # @param {type:"boolean"}
RUN_PRICE_SMA = True # @param {type:"boolean"}
RUN_DUAL_SMA = True # @param {type:"boolean"}
PRICE_SMA_WINDOW = 200 # @param {type:"integer"}
PRICE_ENTRY_CONFIRM_DAYS = 3 # @param {type:"integer"}
PRICE_EXIT_CONFIRM_DAYS = 1 # @param {type:"integer"}
FAST_SMA_WINDOW = 50 # @param {type:"integer"}
SLOW_SMA_WINDOW = 200 # @param {type:"integer"}
CROSS_ENTRY_CONFIRM_DAYS = 1 # @param {type:"integer"}
CROSS_EXIT_CONFIRM_DAYS = 1 # @param {type:"integer"}
COMMISSION_RATE = 0.001 # @param {type:"number"}
TAX_RATE = 0.22 # @param {type:"number"}
EXPENSE_RATIO = 0.0095 # @param {type:"number"}
BORROW_SPREAD = 1.0 # @param {type:"number"}
OUTPUT_NAME = "custom_analysis" # @param {type:"string"}


In [ ]:
from pathlib import Path
import shutil
import subprocess

repo_dir = Path('/content/backtest_repo')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    'git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(repo_dir)
], check=True)

package_dir = repo_dir / PACKAGE_SUBDIR
if not package_dir.exists():
    raise FileNotFoundError(f'Missing package directory: {package_dir}')

print(f'Package directory: {package_dir}')


In [ ]:
import subprocess
from pathlib import Path

package_dir = Path('/content/backtest_repo') / PACKAGE_SUBDIR
cmd = [
    'python', 'run_custom_analysis.py',
    '--start-date', START_DATE,
    '--end-date', END_DATE,
    '--base-series', BASE_SERIES,
    '--leverages', LEVERAGES,
    '--price-sma-window', str(PRICE_SMA_WINDOW),
    '--price-entry-confirm-days', str(PRICE_ENTRY_CONFIRM_DAYS),
    '--price-exit-confirm-days', str(PRICE_EXIT_CONFIRM_DAYS),
    '--fast-sma-window', str(FAST_SMA_WINDOW),
    '--slow-sma-window', str(SLOW_SMA_WINDOW),
    '--cross-entry-confirm-days', str(CROSS_ENTRY_CONFIRM_DAYS),
    '--cross-exit-confirm-days', str(CROSS_EXIT_CONFIRM_DAYS),
    '--commission-rate', str(COMMISSION_RATE),
    '--tax-rate', str(TAX_RATE),
    '--expense-ratio', str(EXPENSE_RATIO),
    '--borrow-spread', str(BORROW_SPREAD),
    '--output-name', OUTPUT_NAME,
]

if INCLUDE_BASE_SERIES:
    cmd.append('--include-base-series')
if INCLUDE_ZERO_COST:
    cmd.append('--include-zero-cost')
if not INCLUDE_COST_ADJUSTED:
    cmd.append('--skip-cost-adjusted')
if not RUN_BUYHOLD:
    cmd.append('--skip-buyhold')
if not RUN_PRICE_SMA:
    cmd.append('--skip-price-sma')
if not RUN_DUAL_SMA:
    cmd.append('--skip-dual-sma')

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=package_dir, check=True)


In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path('/root/backtest_outputs') / OUTPUT_NAME
metrics = pd.read_csv(output_dir / 'metrics.csv')
display(metrics)

config = pd.read_json(output_dir / 'config.json', typ='series')
display(config)


In [ ]:
!cd /root && zip -r /content/backtest_outputs.zip backtest_outputs

from google.colab import files
files.download('/content/backtest_outputs.zip')
